# Exploratory Data Analysis — RetentionIQ

We're analyzing membership data from a **250+ location fitness franchise network in Brazil**. The goal isn't just to describe the data — it's to build intuition for the causal and predictive models that follow. Every pattern we find here informs a modeling decision later.

Specifically, we need answers to four questions before we write a single model:

1. **How do Regular vs Aggregator members differ?** — If the data-generating processes are fundamentally different, pooling them is a modeling mistake.
2. **How heterogeneous are locations?** — If churn rates vary 3x across gyms, a single global model leaves money on the table.
3. **What leading indicators exist for churn?** — Visit frequency trends, tenure effects, seasonal enrollment patterns.
4. **Is there selection bias in historical retention actions?** — If managers already target at-risk members, naive before/after comparisons will be misleading. This determines whether we need causal inference (spoiler: we do).

The data was generated synthetically by `scripts/generate_franchise_data.py` and stored as Parquet in `data/raw/`. Four tables: `locations`, `members`, `visits`, and `retention_actions`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Plotting defaults — clean, publication-ready
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

DATA_DIR = Path("../data/raw/")

# Reference date for tenure/censoring calculations
TODAY = pd.Timestamp("2026-03-17")

In [ ]:
# ── Load all four tables ──────────────────────────────────────────────────────
locations = pd.read_parquet(DATA_DIR / "locations.parquet")
members = pd.read_parquet(DATA_DIR / "members.parquet")
visits = pd.read_parquet(DATA_DIR / "visits.parquet")
actions = pd.read_parquet(DATA_DIR / "retention_actions.parquet")

# Quick date parsing safety — ensure date columns are datetime
for col in ["join_date", "cancel_date"]:
    members[col] = pd.to_datetime(members[col])
visits["visit_date"] = pd.to_datetime(visits["visit_date"])
actions["action_date"] = pd.to_datetime(actions["action_date"])

# Derive tenure in days (right-censored for active members)
members["end_date"] = members["cancel_date"].fillna(TODAY)
members["tenure_days"] = (members["end_date"] - members["join_date"]).dt.days
members["tenure_months"] = members["tenure_days"] / 30.44  # avg days per month

print(f"Locations:         {locations.shape[0]:>8,} rows  x {locations.shape[1]:>2} cols")
print(f"Members:           {members.shape[0]:>8,} rows  x {members.shape[1]:>2} cols")
print(f"Visits:            {visits.shape[0]:>8,} rows  x {visits.shape[1]:>2} cols")
print(f"Retention Actions: {actions.shape[0]:>8,} rows  x {actions.shape[1]:>2} cols")
print(f"\nMembership date range: {members['join_date'].min().date()} → {members['join_date'].max().date()}")
print(f"Visit date range:      {visits['visit_date'].min().date()} → {visits['visit_date'].max().date()}")
print(f"Overall churn rate:    {members['churned'].mean():.1%}")

## 1. Scale & Shape

Before anything else — what are we working with? How much data, how fresh, what's the grain?

Understanding the data shape up front saves hours of debugging later. We need to know: Are there nulls that signal right-censoring (active members with no `cancel_date`) vs actual data quality issues? What's the cardinality of categorical columns? Are there any obvious data generation artifacts we should be aware of?

In [ ]:
# ── Schema overview for each table ────────────────────────────────────────────
for name, df in [("locations", locations), ("members", members), ("visits", visits), ("retention_actions", actions)]:
    print(f"{'=' * 60}")
    print(f"  {name.upper()}  ({df.shape[0]:,} rows x {df.shape[1]} cols)")
    print(f"{'=' * 60}")
    null_counts = df.isnull().sum()
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "nulls": null_counts,
        "null_%": (null_counts / len(df) * 100).round(1),
        "nunique": df.nunique(),
        "example": df.iloc[0],
    })
    print(summary.to_string())
    print()

In [ ]:
# ── Key summary statistics ────────────────────────────────────────────────────
# Note: cancel_date nulls = active members (right-censored), NOT missing data.
active_count = members["churned"].eq(0).sum()
churned_count = members["churned"].eq(1).sum()

summary_stats = pd.DataFrame({
    "Metric": [
        "Total locations",
        "Total members",
        "Active members",
        "Churned members",
        "Overall churn rate",
        "Median tenure (months)",
        "Mean monthly price",
        "Total visits recorded",
        "Total retention actions",
        "Members with ≥1 action",
    ],
    "Value": [
        f"{locations.shape[0]:,}",
        f"{members.shape[0]:,}",
        f"{active_count:,}",
        f"{churned_count:,}",
        f"{members['churned'].mean():.1%}",
        f"{members['tenure_months'].median():.1f}",
        f"R$ {members['monthly_price'].mean():.2f}",
        f"{visits.shape[0]:,}",
        f"{actions.shape[0]:,}",
        f"{actions['member_id'].nunique():,}",
    ]
}).set_index("Metric")

summary_stats.style.set_properties(**{"text-align": "right"})

## 2. Membership Enrollment Patterns

The *when* of enrollment matters more than most people think. January cohorts (New Year's resolutions) churn at dramatically different rates than June cohorts. If we don't account for this in our models, we'll confuse seasonal effects with treatment effects.

This has a direct modeling implication: any train/test split must be **temporal**, not random. A random split would leak future seasonality into the training set and inflate our offline metrics. We enforce temporal splits throughout the pipeline (see `configs/`).

In [ ]:
# ── Monthly enrollment volume + churn rate by enrollment month ────────────────
members["join_month"] = members["join_date"].dt.to_period("M").dt.to_timestamp()
members["join_month_of_year"] = members["join_date"].dt.month

# Enrollments per calendar month
monthly_enrollments = members.groupby("join_month").size().reset_index(name="enrollments")

# Churn rate by month-of-year (seasonal pattern, averaged across years)
churn_by_month = (
    members.groupby("join_month_of_year")["churned"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "churn_rate", "count": "n"})
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: enrollment volume over time
ax1 = axes[0]
ax1.bar(monthly_enrollments["join_month"], monthly_enrollments["enrollments"],
        width=25, color="#4C72B0", alpha=0.8)
ax1.set_title("Monthly Enrollment Volume")
ax1.set_xlabel("Month")
ax1.set_ylabel("New Members")
ax1.tick_params(axis="x", rotation=45)

# Right: churn rate by enrollment month-of-year (dual axis)
ax2 = axes[1]
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
bar_colors = ["#c0392b" if m == 1 else "#4C72B0" for m in churn_by_month["join_month_of_year"]]

bars = ax2.bar(churn_by_month["join_month_of_year"], churn_by_month["n"],
               color=bar_colors, alpha=0.6, label="Enrollment count")
ax2.set_xlabel("Enrollment Month")
ax2.set_ylabel("Enrollment Count", color="#4C72B0")
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(month_labels)

ax2b = ax2.twinx()
ax2b.plot(churn_by_month["join_month_of_year"], churn_by_month["churn_rate"],
          color="#c0392b", marker="o", linewidth=2, label="Churn rate")
ax2b.set_ylabel("Churn Rate", color="#c0392b")
ax2b.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

ax2.set_title("Enrollment Volume & Churn Rate by Month of Year")

# Combined legend
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.tight_layout()
plt.show()

# Quantify the January effect
jan_churn = members.loc[members["join_month_of_year"] == 1, "churned"].mean()
non_jan_churn = members.loc[members["join_month_of_year"] != 1, "churned"].mean()
print(f"January cohort churn rate:     {jan_churn:.1%}")
print(f"Non-January cohort churn rate: {non_jan_churn:.1%}")
print(f"Relative difference:           {(jan_churn / non_jan_churn - 1):+.1%}")

## 3. Regular vs Aggregator: Two Different Businesses

This is arguably the most important split in the entire analysis. Aggregator members (Gympass, TotalPass, Wellhub) behave fundamentally differently from direct members. Pooling them would be a modeling mistake — not because it hurts AUC, but because it **destroys causal interpretability**. The gym has different levers for each segment:

- **Regular members**: The gym controls pricing, communication, and the full relationship. Retention actions (SMS, calls, promos) can be directly attributed.
- **Aggregator members**: The gym is one of many options. The member's loyalty is to the *platform*, not the gym. Price is set by contract with the aggregator. Visit patterns reflect multi-gym usage, not declining interest.

If we build one model and it says "send an SMS to reduce churn," that recommendation is meaningless for an aggregator member whose churn decision depends on their employer's benefits package, not on our outreach.

In [ ]:
# ── Regular vs Aggregator: side-by-side comparison ────────────────────────────

# Compute visits per member per month (approximation using total visits / tenure)
visits_per_member = (
    visits.groupby("member_id")
    .size()
    .reset_index(name="total_visits")
)
members_with_visits = members.merge(visits_per_member, on="member_id", how="left")
members_with_visits["total_visits"] = members_with_visits["total_visits"].fillna(0)
members_with_visits["visits_per_month"] = np.where(
    members_with_visits["tenure_months"] > 0,
    members_with_visits["total_visits"] / members_with_visits["tenure_months"],
    0
)

# Segment comparison table
segment_stats = (
    members_with_visits.groupby("contract_source")
    .agg(
        count=("member_id", "size"),
        churn_rate=("churned", "mean"),
        avg_tenure_months=("tenure_months", "mean"),
        median_tenure_months=("tenure_months", "median"),
        avg_monthly_price=("monthly_price", "mean"),
        avg_visits_per_month=("visits_per_month", "mean"),
    )
    .round(2)
)

# Format for display
display_stats = segment_stats.copy()
display_stats["count"] = display_stats["count"].apply(lambda x: f"{x:,}")
display_stats["churn_rate"] = display_stats["churn_rate"].apply(lambda x: f"{x:.1%}")
display_stats["avg_tenure_months"] = display_stats["avg_tenure_months"].apply(lambda x: f"{x:.1f}")
display_stats["median_tenure_months"] = display_stats["median_tenure_months"].apply(lambda x: f"{x:.1f}")
display_stats["avg_monthly_price"] = display_stats["avg_monthly_price"].apply(lambda x: f"R$ {x:.2f}")
display_stats["avg_visits_per_month"] = display_stats["avg_visits_per_month"].apply(lambda x: f"{x:.1f}")

display_stats.columns = ["Count", "Churn Rate", "Avg Tenure (mo)", "Median Tenure (mo)",
                          "Avg Price", "Avg Visits/Month"]
display_stats

In [ ]:
# ── Tenure distributions by segment ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: overlaid tenure distributions
for source in members["contract_source"].unique():
    subset = members[members["contract_source"] == source]
    axes[0].hist(subset["tenure_months"], bins=50, alpha=0.5, label=source, density=True)
axes[0].set_title("Tenure Distribution by Contract Source")
axes[0].set_xlabel("Tenure (months)")
axes[0].set_ylabel("Density")
axes[0].legend()

# Add median lines
for source, color in zip(members["contract_source"].unique(), ["#4C72B0", "#DD8452"]):
    med = members.loc[members["contract_source"] == source, "tenure_months"].median()
    axes[0].axvline(med, color=color, linestyle="--", alpha=0.8)
    axes[0].text(med + 0.3, axes[0].get_ylim()[1] * 0.9,
                 f"median={med:.0f}mo", color=color, fontsize=9)

# Right: visits/month distribution by segment
for source in members_with_visits["contract_source"].unique():
    subset = members_with_visits[members_with_visits["contract_source"] == source]
    # Cap at 99th percentile for visualization
    cap = subset["visits_per_month"].quantile(0.99)
    axes[1].hist(subset["visits_per_month"].clip(upper=cap), bins=50,
                 alpha=0.5, label=source, density=True)
axes[1].set_title("Visit Frequency Distribution by Contract Source")
axes[1].set_xlabel("Visits per Month")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Aggregator platform breakdown (for aggregator members only) ───────────────
agg_members = members[members["contract_source"] == "aggregator"]

if "aggregator_platform" in agg_members.columns and agg_members["aggregator_platform"].notna().any():
    platform_stats = (
        agg_members.groupby("aggregator_platform")
        .agg(
            count=("member_id", "size"),
            churn_rate=("churned", "mean"),
            avg_tenure=("tenure_months", "mean"),
        )
        .sort_values("count", ascending=False)
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    x = range(len(platform_stats))
    bars = ax.bar(x, platform_stats["count"], color="#4C72B0", alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(platform_stats.index, rotation=0)
    ax.set_ylabel("Member Count")
    ax.set_title("Aggregator Platform Distribution")

    # Annotate with churn rate
    for i, (_, row) in enumerate(platform_stats.iterrows()):
        ax.text(i, row["count"] + max(platform_stats["count"]) * 0.02,
                f"churn: {row['churn_rate']:.0%}", ha="center", fontsize=9, color="#c0392b")

    plt.tight_layout()
    plt.show()
else:
    print("No aggregator_platform column or all values null — skipping breakdown.")

## 4. Location Heterogeneity

250 locations are not 250 copies of the same gym. Urban locations tend to have higher capacity but often higher churn — more competition, more alternatives, less community lock-in. Regional differences matter too: economic conditions, climate (outdoor alternatives), and local culture all affect retention.

This heterogeneity is why we need a **per-location budget optimizer** rather than a one-size-fits-all retention strategy. A location with 15% churn and R$50k monthly revenue needs a fundamentally different intervention budget than one with 40% churn and R$15k revenue. The optimization layer (see `src/optimization/`) allocates budget across locations to maximize retained revenue, not just minimize average churn.

In [ ]:
# ── Location-level churn rates ────────────────────────────────────────────────
loc_churn = (
    members.groupby("location_id")
    .agg(
        member_count=("member_id", "size"),
        churn_rate=("churned", "mean"),
    )
    .reset_index()
    .merge(locations, on="location_id", how="left")
)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Top-left: distribution of churn rates across locations
ax = axes[0, 0]
ax.hist(loc_churn["churn_rate"], bins=30, color="#4C72B0", alpha=0.8, edgecolor="white")
ax.axvline(loc_churn["churn_rate"].mean(), color="#c0392b", linestyle="--", linewidth=2,
           label=f"Mean: {loc_churn['churn_rate'].mean():.1%}")
ax.axvline(loc_churn["churn_rate"].median(), color="#2ecc71", linestyle="--", linewidth=2,
           label=f"Median: {loc_churn['churn_rate'].median():.1%}")
ax.set_title("Distribution of Churn Rates Across Locations")
ax.set_xlabel("Location Churn Rate")
ax.set_ylabel("Number of Locations")
ax.legend()
ax.text(0.98, 0.85,
        f"Range: {loc_churn['churn_rate'].min():.0%} – {loc_churn['churn_rate'].max():.0%}\n"
        f"Std:   {loc_churn['churn_rate'].std():.1%}",
        transform=ax.transAxes, ha="right", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

# Top-right: urban vs rural
ax = axes[0, 1]
urban_data = [loc_churn.loc[loc_churn["is_urban"] == True, "churn_rate"],
              loc_churn.loc[loc_churn["is_urban"] == False, "churn_rate"]]
bp = ax.boxplot(urban_data, labels=["Urban", "Rural"], patch_artist=True,
                boxprops=dict(alpha=0.7))
bp["boxes"][0].set_facecolor("#4C72B0")
bp["boxes"][1].set_facecolor("#55A868")
ax.set_title("Churn Rate: Urban vs Rural Locations")
ax.set_ylabel("Churn Rate")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

for i, data in enumerate(urban_data, 1):
    ax.text(i, ax.get_ylim()[1] * 0.95, f"n={len(data)}", ha="center", fontsize=10)

# Bottom-left: regional breakdown
ax = axes[1, 0]
region_stats = (
    loc_churn.groupby("region")
    .agg(
        mean_churn=("churn_rate", "mean"),
        n_locations=("location_id", "size"),
    )
    .sort_values("mean_churn", ascending=True)
    .reset_index()
)
bars = ax.barh(region_stats["region"], region_stats["mean_churn"], color="#4C72B0", alpha=0.8)
ax.set_xlabel("Mean Churn Rate")
ax.set_title("Average Churn Rate by Region")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

for i, (_, row) in enumerate(region_stats.iterrows()):
    ax.text(row["mean_churn"] + 0.005, i,
            f"  {row['mean_churn']:.1%} (n={row['n_locations']})", va="center", fontsize=9)

# Bottom-right: capacity vs churn rate
ax = axes[1, 1]
scatter = ax.scatter(loc_churn["capacity"], loc_churn["churn_rate"],
                     c=loc_churn["is_urban"].astype(int), cmap="coolwarm",
                     alpha=0.6, s=40, edgecolors="white", linewidth=0.5)
ax.set_xlabel("Location Capacity")
ax.set_ylabel("Churn Rate")
ax.set_title("Capacity vs Churn Rate (colored by Urban/Rural)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

# Add correlation
corr = loc_churn["capacity"].corr(loc_churn["churn_rate"])
ax.text(0.02, 0.95, f"Pearson r = {corr:.3f}", transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

# Legend for scatter colors
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker="o", color="w", markerfacecolor="#3b4cc0",
                          markersize=8, label="Rural"),
                   Line2D([0], [0], marker="o", color="w", markerfacecolor="#b40426",
                          markersize=8, label="Urban")]
ax.legend(handles=legend_elements, loc="upper right")

plt.tight_layout()
plt.show()

print(f"\nLocation churn rate summary:")
print(f"  Min:    {loc_churn['churn_rate'].min():.1%}")
print(f"  Q1:     {loc_churn['churn_rate'].quantile(0.25):.1%}")
print(f"  Median: {loc_churn['churn_rate'].median():.1%}")
print(f"  Q3:     {loc_churn['churn_rate'].quantile(0.75):.1%}")
print(f"  Max:    {loc_churn['churn_rate'].max():.1%}")
print(f"  IQR:    {loc_churn['churn_rate'].quantile(0.75) - loc_churn['churn_rate'].quantile(0.25):.1%}")

## 5. Visit Patterns & Early Warning Signals

Visit frequency is the strongest leading indicator of churn — but not in the way a naive model would capture it. A member going from 12 visits/month to 4 is very different from a member who's always visited 4 times. **The trend matters more than the level.** This is why we engineer `visit_frequency_trend` as a feature rather than just using raw visit counts.

There's a subtlety here that's easy to miss: visit frequency naturally decays with tenure for *all* members, even ones who don't churn. New members are excited; six months in, life happens. The signal we care about is **excess decay** — visits declining faster than the population norm for that tenure. This is why we'll compute visit frequency relative to cohort-and-tenure benchmarks in the feature engineering stage.

In [ ]:
# ── Visit frequency over tenure, split by segment and churn outcome ──────────

# For each visit, compute the member's tenure at the time of visit
visits_enriched = visits.merge(
    members[["member_id", "join_date", "contract_source", "churned"]],
    on="member_id",
    how="left"
)
visits_enriched["tenure_at_visit_months"] = (
    (visits_enriched["visit_date"] - visits_enriched["join_date"]).dt.days / 30.44
).clip(lower=0)
visits_enriched["tenure_bucket"] = visits_enriched["tenure_at_visit_months"].apply(
    lambda x: int(x) if x < 24 else 24  # bucket by month, cap at 24
)

# Visits per member per tenure-month
visit_counts = (
    visits_enriched.groupby(["member_id", "contract_source", "churned", "tenure_bucket"])
    .size()
    .reset_index(name="visits_in_month")
)

# Average across members for each segment x churn status x tenure bucket
avg_visits = (
    visit_counts.groupby(["contract_source", "churned", "tenure_bucket"])["visits_in_month"]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, source in enumerate(["regular", "aggregator"]):
    ax = axes[idx]
    for churned_val, label, color, ls in [(0, "Active", "#2ecc71", "-"), (1, "Churned", "#c0392b", "--")]:
        subset = avg_visits[
            (avg_visits["contract_source"] == source) &
            (avg_visits["churned"] == churned_val) &
            (avg_visits["tenure_bucket"] <= 18)  # enough data for first 18 months
        ].sort_values("tenure_bucket")
        ax.plot(subset["tenure_bucket"], subset["visits_in_month"],
                color=color, linestyle=ls, linewidth=2, marker="o", markersize=4, label=label)

    ax.set_title(f"Visit Frequency Over Tenure — {source.title()} Members")
    ax.set_xlabel("Tenure (months)")
    ax.set_ylabel("Avg Visits per Month")
    ax.legend()
    ax.set_xlim(0, 18)

plt.tight_layout()
plt.show()

In [ ]:
# ── Days since last visit: churned vs active ─────────────────────────────────
# For active members, "days since last visit" = how recently they came in
# For churned members, we measure from their last visit to their cancel_date

last_visit = visits.groupby("member_id")["visit_date"].max().reset_index(name="last_visit_date")
members_lv = members.merge(last_visit, on="member_id", how="left")

# For churned: days between last visit and cancellation
# For active: days between last visit and today
members_lv["days_since_last_visit"] = np.where(
    members_lv["churned"] == 1,
    (members_lv["cancel_date"] - members_lv["last_visit_date"]).dt.days,
    (TODAY - members_lv["last_visit_date"]).dt.days
)

# Filter out members with no visits (they'll have NaN)
members_lv = members_lv.dropna(subset=["days_since_last_visit"])
members_lv["days_since_last_visit"] = members_lv["days_since_last_visit"].clip(lower=0)

fig, ax = plt.subplots(figsize=(12, 5))

for churned_val, label, color in [(0, "Active", "#2ecc71"), (1, "Churned", "#c0392b")]:
    subset = members_lv[members_lv["churned"] == churned_val]["days_since_last_visit"]
    # Cap at 180 days for visualization
    ax.hist(subset.clip(upper=180), bins=60, alpha=0.5, color=color, label=label, density=True)
    med = subset.median()
    ax.axvline(med, color=color, linestyle="--", linewidth=1.5, alpha=0.8)
    ax.text(med + 2, ax.get_ylim()[1] * 0.85 if churned_val == 0 else ax.get_ylim()[1] * 0.75,
            f"median={med:.0f}d", color=color, fontsize=10)

ax.set_title("Days Since Last Visit: Active vs Churned Members")
ax.set_xlabel("Days Since Last Visit (capped at 180)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Active members — median days since last visit: {members_lv.loc[members_lv['churned'] == 0, 'days_since_last_visit'].median():.0f}")
print(f"Churned members — median days between last visit and cancel: {members_lv.loc[members_lv['churned'] == 1, 'days_since_last_visit'].median():.0f}")

In [ ]:
# ── Visit heatmap: day of week ────────────────────────────────────────────────
visits_enriched["day_of_week"] = visits_enriched["visit_date"].dt.dayofweek
visits_enriched["month"] = visits_enriched["visit_date"].dt.month

# Average daily visits by day of week and month
dow_month = (
    visits_enriched.groupby(["month", "day_of_week"])
    .size()
    .reset_index(name="visit_count")
    .pivot(index="day_of_week", columns="month", values="visit_count")
    .fillna(0)
)

day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
month_labels_short = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                      "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(dow_month, annot=True, fmt=".0f", cmap="YlOrRd",
            xticklabels=month_labels_short[:dow_month.shape[1]],
            yticklabels=day_labels, ax=ax, linewidths=0.5)
ax.set_title("Visit Volume Heatmap: Day of Week x Month")
ax.set_xlabel("Month")
ax.set_ylabel("Day of Week")
plt.tight_layout()
plt.show()

## 6. Retention Actions: The Selection Bias Problem

Here's the pattern that makes this project interesting from a causal inference perspective. Look at the data: members who received retention actions have **higher** churn rates than those who didn't. Does this mean retention actions *cause* churn? Obviously not.

Managers target at-risk members — creating **selection bias**. The members who receive a "win-back" SMS were already on the way out. A naive model would learn that "sending an SMS increases churn probability." This is Simpson's paradox in action, and it's exactly why we need causal inference (notebooks 03-04) rather than just predictive modeling.

The key question isn't *whether* retention actions work — it's *which* actions work for *which* members, and by how much. That requires estimating the **Average Treatment Effect on the Treated (ATT)**, controlling for the confounders that drove the manager's targeting decision in the first place.

In [ ]:
# ── The selection bias table: churn rate by action receipt ────────────────────
members_acted = members["member_id"].isin(actions["member_id"].unique())
members["received_action"] = members_acted

bias_table = (
    members.groupby("received_action")
    .agg(
        count=("member_id", "size"),
        churn_rate=("churned", "mean"),
        avg_tenure_months=("tenure_months", "mean"),
    )
    .rename(index={True: "Received Action", False: "No Action"})
)

print("=== The Naive (Wrong) Comparison ===\n")
print(bias_table.to_string(
    formatters={
        "count": lambda x: f"{x:,}",
        "churn_rate": lambda x: f"{x:.1%}",
        "avg_tenure_months": lambda x: f"{x:.1f}",
    }
))
print(f"\nNaive difference in churn: "
      f"{bias_table.loc['Received Action', 'churn_rate'] - bias_table.loc['No Action', 'churn_rate']:+.1%}")
print("(This is MISLEADING — it reflects selection bias, not causal effect)")

In [ ]:
# ── Action type distribution and timing ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: action type counts
action_counts = actions["action_type"].value_counts()
ax = axes[0]
bars = ax.barh(action_counts.index, action_counts.values, color="#4C72B0", alpha=0.8)
ax.set_xlabel("Number of Actions")
ax.set_title("Retention Action Types")
for i, (action, count) in enumerate(action_counts.items()):
    ax.text(count + max(action_counts) * 0.01, i, f"{count:,}", va="center", fontsize=10)

# Right: timing — when do actions happen relative to churn?
# For churned members who received actions, compute days between action and cancel_date
churned_with_actions = actions.merge(
    members[members["churned"] == 1][["member_id", "cancel_date"]],
    on="member_id",
    how="inner"
)
churned_with_actions["days_before_churn"] = (
    churned_with_actions["cancel_date"] - churned_with_actions["action_date"]
).dt.days

ax = axes[1]
valid_timing = churned_with_actions["days_before_churn"].clip(lower=-30, upper=180)
ax.hist(valid_timing, bins=50, color="#c0392b", alpha=0.7, edgecolor="white")
ax.axvline(0, color="black", linestyle="-", linewidth=1.5, label="Churn date")
ax.axvline(valid_timing.median(), color="#4C72B0", linestyle="--", linewidth=1.5,
           label=f"Median: {valid_timing.median():.0f} days before")
ax.set_title("When Are Actions Taken Relative to Churn?")
ax.set_xlabel("Days Before Churn (positive = before cancellation)")
ax.set_ylabel("Number of Actions")
ax.legend()

plt.tight_layout()
plt.show()

print(f"Median action timing: {churned_with_actions['days_before_churn'].median():.0f} days before churn")
print(f"Actions taken after churn: {(churned_with_actions['days_before_churn'] < 0).sum():,} "
      f"({(churned_with_actions['days_before_churn'] < 0).mean():.1%})")

In [ ]:
# ── The confounding visual: action recipients had LOWER visit frequency ──────
# This is the smoking gun for selection bias. Members who received actions
# were already visiting less frequently — that's WHY they were targeted.

# Compute avg visits/month in the 60 days BEFORE the first action (for action recipients)
first_action = actions.groupby("member_id")["action_date"].min().reset_index(name="first_action_date")

# For action recipients: visits in 60 days before first action
members_first_action = members.merge(first_action, on="member_id", how="left")

# Pre-action visit rate for members who received actions
action_members = members_first_action[members_first_action["first_action_date"].notna()].copy()
pre_action_visits = visits.merge(
    action_members[["member_id", "first_action_date"]],
    on="member_id",
    how="inner"
)
pre_action_visits = pre_action_visits[
    (pre_action_visits["visit_date"] >= pre_action_visits["first_action_date"] - pd.Timedelta(days=60)) &
    (pre_action_visits["visit_date"] < pre_action_visits["first_action_date"])
]
pre_action_rate = (
    pre_action_visits.groupby("member_id").size().reset_index(name="visits_60d")
)
pre_action_rate["visits_per_month_pre"] = pre_action_rate["visits_60d"] / 2.0  # 60 days ~ 2 months

# For non-action members: overall visit rate (as comparison baseline)
no_action_members = members_with_visits[~members_with_visits["received_action"]].copy()

fig, ax = plt.subplots(figsize=(10, 5))

# Action recipients: pre-action visit rate
action_rate_data = pre_action_rate["visits_per_month_pre"].clip(upper=30)
no_action_rate_data = no_action_members["visits_per_month"].clip(upper=30)

ax.hist(no_action_rate_data, bins=40, alpha=0.5, color="#2ecc71", density=True,
        label=f"No Action (mean: {no_action_rate_data.mean():.1f}/mo)")
ax.hist(action_rate_data, bins=40, alpha=0.5, color="#c0392b", density=True,
        label=f"Pre-Action Period (mean: {action_rate_data.mean():.1f}/mo)")

ax.set_title("The Confound: Visit Frequency Before Action vs Never-Treated Members")
ax.set_xlabel("Visits per Month")
ax.set_ylabel("Density")
ax.legend(fontsize=11)

ax.text(0.98, 0.65,
        "Action recipients were already\nvisiting LESS frequently.\n"
        "This is why naive comparison\nis biased.",
        transform=ax.transAxes, ha="right", fontsize=10, style="italic",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#fff3cd", alpha=0.9))

plt.tight_layout()
plt.show()

## 7. Key Takeaways for Modeling

This EDA isn't just descriptive — every finding maps directly to an architectural decision in the RetentionIQ pipeline:

1. **Separate models for Regular vs Aggregator members.** The data-generating processes are fundamentally different. Pooling them would produce a model that's wrong for both segments. We train segment-specific survival models in `src/models/` and maintain separate feature stores in Feast.

2. **Temporal features are critical.** Enrollment month, visit frequency *trends* (not levels), and seasonality all carry strong signal. Any feature engineering that ignores time-ordering will leak information. All train/test splits in the pipeline are temporal — no exceptions.

3. **Selection bias in retention actions demands causal inference.** The naive data says actions "increase" churn. This is confounding, not causation. We use DoWhy for DAG specification, EconML for heterogeneous treatment effect estimation, and causal forests to identify which members would benefit most from each intervention type. See `src/causal/`.

4. **Location heterogeneity requires per-location optimization.** A single retention budget rule would over-invest in low-churn locations and under-invest in high-churn ones. The Pyomo optimizer in `src/optimization/` allocates intervention budget per location to maximize expected retained LTV, subject to operational constraints (staff hours, budget caps).

5. **Right-censoring is real and pervasive.** Active members haven't churned *yet* — they're censored observations, not negative examples. Treating them as "not churned" in a binary classifier would bias the model toward recent joiners. This is why survival analysis (Cox PH, accelerated failure time models) is the right framework for the primary churn model.

---

**Next notebooks:**
- `02_feature_engineering.ipynb` — Build the feature set, including temporal visit trends and cohort-relative metrics
- `03_causal_dag.ipynb` — Specify the causal DAG with DoWhy and validate with data
- `04_treatment_effects.ipynb` — Estimate heterogeneous treatment effects with EconML